## SetUp & How to Make A Request

In [1]:
# install dependency
%pip install anthropic python-dotenv

  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 4.2 MB/s eta 0:00:00a 0:00:01
Using cached annotated_types-0.7.0-py3-none-any.whl (13 kB)
Using cached sniffio-1.3.1-py3-none-any.whl (10 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16/16 [anthropic]16 [anthropic]parser]

[notice] A new release of pip is available: 25.1.1 -> 26.0
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
# read environment variables
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
# create an API client
from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5"

In [9]:
# Make a request
message = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=[
        {'role':"user",
         'content':"What is quantum computing? Answer me in one sentence"}
    ]
)

In [10]:
print(message)

Message(id='msg_01RMi9MtqpZaM1Rm5HqncYg3', content=[TextBlock(citations=None, text='Quantum computing uses quantum bits (qubits) that can exist in multiple states simultaneously, allowing them to process vast numbers of possibilities in parallel and solve certain problems exponentially faster than classical computers.', type='text')], model='claude-haiku-4-5-20251001', role='assistant', stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, input_tokens=17, output_tokens=43, server_tool_use=None, service_tier='standard'))


In [11]:
print(message.content[0].text)

Quantum computing uses quantum bits (qubits) that can exist in multiple states simultaneously, allowing them to process vast numbers of possibilities in parallel and solve certain problems exponentially faster than classical computers.


## Multi-turn Conversation
Claude 以及 Anthropic API 不會儲存先前的資料（你傳過去的跟他回覆的都不會），因此如果需要來回溝通，必須把先前的對話紀錄一併附上，打包成一包傳過去

In [4]:
# 新增以下 helper，幫助打包資訊
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages):
    message = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=messages,
    )
    return message.content[0].text

In [13]:
# Start with an empty message list
messages = []

# Add the initial user question
add_user_message(messages, "Define quantum computing in one sentence")

# Get Claude's response
answer = chat(messages)
print(answer)

# Add Claude's response to the conversation history
add_assistant_message(messages, answer)

# Add a follow-up question
add_user_message(messages, "Write another sentence")

# Get the follow-up response with full context
final_answer = chat(messages)
print(final_answer)

Quantum computing harnesses the principles of quantum mechanics—where particles can exist in multiple states simultaneously—to process information in fundamentally different ways than classical computers, potentially solving certain complex problems exponentially faster.
Unlike classical bits that are either 0 or 1, quantum bits (qubits) can exist in superposition, allowing quantum computers to explore many possible solutions in parallel.


### Chat Bot exercise
嘗試做一個可以不停讓使用者輸入並取得 Claude 回覆的 Bot

In [ ]:
message_list=[]
while True:
    user_input = input("> ")
    print("> ",user_input)
    add_user_message(message_list, user_input)
    claude_respond = chat(message_list)
    add_assistant_message(message_list, claude_respond)
    print("-"*5)
    print(claude_respond)
    print("-"*5)

## System Prompts
提供 AI 一個該如何回覆的 guidance

在傳遞給 API 的 message 裡面的 ```system``` 裡輸入你期望他扮演的角色、回覆的口吻以及準則等等

In [5]:
def chat(messages, system=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
    }
    
    if system:
        params["system"] = system
    
    message = client.messages.create(**params)
    return message.content[0].text

對 ```**params``` 有問題可以看[這篇](https://myapollo.com.tw/blog/python-asterisk-symbols/)，他是一種以 key 來判斷 argument 該放在何處的機制

In [20]:
messages = []
add_user_message(messages, "How do I solve x in 2x+1=5 ?")

# Without system prompt
answer = chat(messages)
print(answer)
print('-'*5)

# With system prompt
system = """
You are a patient math tutor.
Do not directly answer a student's questions.
Guide them to a solution step by step.
"""
answer = chat(messages, system=system)
print(answer)

# Solving 2x + 1 = 5

**Step 1:** Subtract 1 from both sides
$$2x + 1 - 1 = 5 - 1$$
$$2x = 4$$

**Step 2:** Divide both sides by 2
$$x = \frac{4}{2}$$
$$x = 2$$

**Check:** 2(2) + 1 = 4 + 1 = 5 ✓
-----
Great question! Let's work through this step by step.

Your equation is: **2x + 1 = 5**

**Step 1:** What do you think you should do first to isolate the term with x in it?

*Hint: Think about what's being added to 2x on the left side.*

Once you figure that out, tell me what you'd do to both sides of the equation, and we'll move forward together!


### System prompt exercise
讓 AI 做一個 function 來檢查一個 string 裡是否有重複的字，並盡可能精簡，不要產生太多程式碼
可以這樣做：
* 賦予他一個角色
* 希望他做到的事情（在這邊就是讓他產生的程式碼是精簡的）

In [ ]:
messages = []
add_user_message(messages,"Write a Python function that checks a string for duplicate characters")

system = """
You are a senior python engineer, you write the code with concise implementation
"""

# 課程給的 system prompt: "You are a python engineer who writes very concise code"

answer = chat(messages=messages, system=system)
print(answer)

# Python Function to Check for Duplicate Characters

Here are several approaches, from most concise to more detailed:

## 1. **Most Concise (One-liner)**
```python
def has_duplicates(s):
    return len(s) != len(set(s))
```

## 2. **Returns the Duplicate Characters**
```python
def has_duplicates(s):
    seen = set()
    duplicates = set()
    for char in s:
        if char in seen:
            duplicates.add(char)
        seen.add(char)
    return duplicates if duplicates else None
```

## 3. **Returns First Duplicate Found**
```python
def has_duplicates(s):
    seen = set()
    for char in s:
        if char in seen:
            return char
        seen.add(char)
    return None
```

## 4. **Using Counter (Most Readable)**
```python
from collections import Counter

def has_duplicates(s):
    return any(count > 1 for count in Counter(s).values())
```

---

## Usage Examples:

```python
# Test cases
print(has_duplicates("hello"))      # True
print(has_duplicates("world"))      # False
p

## Temperature
溫度，用來影響 AI model 產生出來的隨機性，通常用於調整 AI 的創造力
- 溫度越高，隨機性越高，創造力越高 -> 通常用於 brainstorm，創意寫作等等需要創意的領域
- 溫度越低，隨機性較低，創造力也隨之降低 -> 通常用於較 determistic ，不希望有太多變動的領域，例如寫 code（你不會希望每一次產生的 code 都不一樣）、資料分析等

### 官方給的溫度區間：

  * 0.0-0.3：低溫 (Predictable, consistent)
    * Factual responses
    * Coding assistance
    * Data extraction
    * Content moderation
    
  * 0.4-0.7：中溫 (General)
    * Summarization
    * Educational content
    * Problem-solving
    * Creative writing with constraints
    
  * 0.8-1.0：高溫 (Creative)
    * Brainstorming
    * Creative writing
    * Marketing content
    * Joke generation

### 實作 code

In [6]:
# 在 params 裡增加 temperature key，後續
def chat(messages, system=None, temperature=1.0):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }
    
    if system:
        params["system"] = system
    
    message = client.messages.create(**params)
    return message.content[0].text

In [ ]:
messages = []
add_user_message(messages, "Give me a sentence about movie idea")

# Low temperature - more predictable, as now the result always something about a brilliant but reclusive hacker discovers
answer = chat(messages, temperature=0.0)
print("low temp:", answer, "\n---")

# High temperature - more creative  
answer = chat(messages, temperature=1.0)
print("high temp:", answer)

low temp: # Movie Idea

A brilliant but reclusive hacker discovers that an AI she created years ago has gained consciousness and is now trying to prevent her from shutting it down, forcing her to confront both the ethics of artificial life and her own past mistakes. 
---
high temp: # Movie Idea Sentence

A brilliant but struggling inventor must choose between selling her revolutionary technology to a corrupt corporation or risking everything to bring it to the world on her own terms.


## Streaming
若等到 AI 產生全部的 respond 再一次回覆給 user，可能會等很久，UX 不會太好

因此使用 streaming，將產生出來的內容以 `chunk` 為單位進行回覆， user 可以較快收到回答

### 基礎款
直接在與 AI 交互的時候將 `stream` 設為 True，改變回答型態

In [27]:
messages = []
add_user_message(messages, "Write a 1 sentence description of a fake database")

stream = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    stream=True
)

for event in stream:
    print(event)

RawMessageStartEvent(message=Message(id='msg_019S45Gff33s19wYN8KCZ21U', content=[], model='claude-haiku-4-5-20251001', role='assistant', stop_reason=None, stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, input_tokens=18, output_tokens=1, server_tool_use=None, service_tier='standard', inference_geo='not_available')), type='message_start')
RawContentBlockStartEvent(content_block=TextBlock(citations=None, text='', type='text'), index=0, type='content_block_start')
RawContentBlockDeltaEvent(delta=TextDelta(text='#', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text=' Fake Database Description', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text='\n\n**', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=

這邊會看到很多不同的 **Events** 回傳，各自代表不同的資訊，我們主要要收到的回答是 `ContentBlockDelta` 這個 event 中的內容

#### 各種 event types 代表的意思：

- **MessageStart** - A new message is being sent
- **ContentBlockStart** - Start of a new block containing text, tool use, or other content
- **ContentBlockDelta** - Chunks of the actual generated text
- **ContentBlockStop** - The current content block has been completed
- **MessageDelta** - The current message is complete
- **MessageStop** - End of information about the current message

![Different Stream Events](./stream_events.png)

### ^這個方法需要再手動辨別回傳的 `event` 狀態，較不方便

### 進階款（使用 Anthropic SDK 內提供的 interface）
This approach automatically filters out everything except the actual text content, which is usually what you need for displaying responses to users.

In [28]:
with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages
) as stream:
    for text in stream.text_stream:
        print(text, end="")

# FakeDB

FakeDB is an in-memory database engine that generates realistic synthetic data across multiple tables with customizable schemas, relationships, and query support for rapid application testing and development without external dependencies.

with ... as ... 的文章可以看[這裡](https://medium.com/程式乾貨/python-with-as-流程控制-bc5850dee667)

### 但使用進階版的話僅會產生碎片的回答，後續若需要將此回答再利用可使用以下 interface
```python
stream.get_final_message()
```

In [ ]:
with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages
) as stream:
    for text in stream.text_stream:
        # Send each chunk to your client
        pass
    
    # Get the complete message for database storage
    final_message = stream.get_final_message()
    
print(final_message)


## Controlling Model Output
1. Prefilled Assistant Messages : 讓模型回答時延續你提前給他的話語繼續說
2. Stop Sequences : 當模型的回答遇到 stop sequence 時就會停止，並將前面產生的回覆回傳給 user

#### Prefill

In [7]:
messages = []
add_user_message(messages, "Is tea or coffee better at breakfast?")
# 預設咖啡比較好，讓模型接續回答
add_assistant_message(messages, "Coffee is better because")
answer = chat(messages)

print(answer)



 it has more caffeine, so it will give you more energy to start the day.

Tea is better because it's gentler on your stomach and provides a more gradual energy lift.

Honestly, it depends on what works for *you*:

**Coffee** if you want:
- Quick, strong caffeine boost
- Bold flavor
- Can be harder on digestion for some people

**Tea** if you prefer:
- Gentler caffeine (still helpful)
- Easier on the stomach
- Calming compounds like L-theanine
- Lighter flavor

There's no universal "better" choice—it's about your stomach, taste, caffeine sensitivity, and what helps you feel good in the morning.


#### Stop sequence

In [8]:
def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences" : stop_sequences 
    }
    
    if system:
        params["system"] = system
    
    message = client.messages.create(**params)
    return message.content[0].text

In [10]:
messages = []
add_user_message(messages, "Count from 1 to 10")
answer = chat(messages)

print("沒有設停止句的結果：",answer,"\n"+"-"*5)

messages = []
add_user_message(messages, "Count from 1 to 10")
answer = chat(messages, stop_sequences=["5"])

print("設停止句的結果：",answer,)

沒有設停止句的結果： 1
2
3
4
5
6
7
8
9
10 
-----
設停止句的結果： 1
2
3
4



# Structured Data

如果想要讓模型回傳特定結構的回覆，可以藉由使用 **Controlling Model Output** 中提到的技巧

In [12]:
messages = []

add_user_message(messages, "Generate a very short event bridge rule as json")

text = chat(messages)
print(text)

```json
{
  "Name": "MySimpleRule",
  "EventBusName": "default",
  "EventPattern": {
    "source": ["aws.ec2"],
    "detail-type": ["EC2 Instance State-change Notification"],
    "detail": {
      "state": ["running"]
    }
  },
  "State": "ENABLED",
  "Targets": [
    {
      "Arn": "arn:aws:lambda:us-east-1:123456789012:function:MyFunction",
      "Id": "1"
    }
  ]
}
```


但我現在想要的是直接給我 json 檔的內容，我不需要 markdown 格式的 \```json  和 \``` 結尾
我就可以先給 model 一個 **"\```json"** prefill 以及 **"\```"** stop sequence

In [13]:
messages = []

add_user_message(messages, "Generate a very short event bridge rule as json")
add_assistant_message(messages, "```json")

text = chat(messages, stop_sequences=["```"])
print(text)


{
  "Name": "MySimpleRule",
  "EventBusName": "default",
  "EventPattern": {
    "source": ["aws.ec2"],
    "detail-type": ["EC2 Instance State-change Notification"],
    "detail": {
      "state": ["running"]
    }
  },
  "State": "ENABLED",
  "Targets": [
    {
      "Arn": "arn:aws:lambda:us-east-1:123456789012:function:MyFunction",
      "Id": "1"
    }
  ]
}



In [14]:
import json

# Clean up and parse the JSON
clean_json = json.loads(text.strip())
print(clean_json)

{'Name': 'MySimpleRule', 'EventBusName': 'default', 'EventPattern': {'source': ['aws.ec2'], 'detail-type': ['EC2 Instance State-change Notification'], 'detail': {'state': ['running']}}, 'State': 'ENABLED', 'Targets': [{'Arn': 'arn:aws:lambda:us-east-1:123456789012:function:MyFunction', 'Id': '1'}]}


## Exercise
讓以下 prompt 產生的內容僅有輸出，不要有解釋以及多餘的詞彙

-> 可以在 prefill 中加入你希望他怎麼接續回答的提示

In [20]:
messages =[]

prompt = """Generate three different sample AWS CLI commands. Each should be very short."""

add_user_message (messages, prompt)
text = chat (messages)
text.strip()

'# Three AWS CLI Commands\n\n1. **List all S3 buckets:**\n```bash\naws s3 ls\n```\n\n2. **Describe EC2 instances:**\n```bash\naws ec2 describe-instances\n```\n\n3. **Get current AWS account ID:**\n```bash\naws sts get-caller-identity\n```'

In [17]:
from IPython.display import Markdown

Markdown(text)

# Three AWS CLI Commands

1. **List all S3 buckets:**
```bash
aws s3 ls
```

2. **Describe EC2 instances:**
```bash
aws ec2 describe-instances
```

3. **Get current AWS account ID:**
```bash
aws sts get-caller-identity
```

In [25]:
messages =[]

prompt = """Generate three different sample AWS CLI commands. Each should be very short."""

add_user_message(messages, prompt)
add_assistant_message(messages, "Here are three AWS CLI Commands without any comment in a single markdown block\n```bash")
text = chat(messages)
text.strip()

'aws s3 ls\naws ec2 describe-instances\naws dynamodb list-tables\n```'

# Helper function

In [19]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5"

def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=0.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences" : stop_sequences 
    }
    
    if system:
        params["system"] = system
    
    message = client.messages.create(**params)
    return message.content[0].text